# Liveliness Detection Challenge

In [22]:
import cv2
import mediapipe as mp
import random
import time
from scipy.spatial import distance

LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

In [23]:
def eye_aspect_ratio(eye):
    A = distance.euclidean(eye[1], eye[5])
    B = distance.euclidean(eye[2], eye[4])
    C = distance.euclidean(eye[0], eye[3])
    return (A + B) / (2 * C)

def get_eye_points(face_landmarks, indices, w, h):
    points = []
    for idx in indices:
        lm = face_landmarks.landmark[idx]
        points.append((int(lm.x * w), int(lm.y * h)))
    return points

def get_horizontal_ratio(point, boundary_1, boundary_2):
    screen_left = min(boundary_1.x, boundary_2.x)
    screen_right = max(boundary_1.x, boundary_2.x)
    dist_to_left = abs(point.x - screen_left)
    dist_to_right = abs(point.x - screen_right)
    if dist_to_left + dist_to_right == 0:
        return 0.5
    return dist_to_left / (dist_to_left + dist_to_right)

def check_ratios(face):
    ratio_R = get_horizontal_ratio(face.landmark[468], face.landmark[33], face.landmark[133])
    ratio_L = get_horizontal_ratio(face.landmark[473], face.landmark[362], face.landmark[263])
    gaze_ratio = (ratio_R + ratio_L) / 2
    
    nose_ratio = get_horizontal_ratio(face.landmark[1], face.landmark[234], face.landmark[454])
    return gaze_ratio, nose_ratio

In [24]:
def detect_head_left(TIME_LIMIT):
    print("Booting camera for Left Detection... 💀")
    cap = cv2.VideoCapture(0)
    mp_face_mesh = mp.solutions.face_mesh
    face_mesh = mp_face_mesh.FaceMesh(refine_landmarks=True, min_detection_confidence=0.5, min_tracking_confidence=0.5)

    success_time = None
    start_time = time.time()

    while time.time() - start_time < TIME_LIMIT:
        ret, frame = cap.read()
        if not ret: break

        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = face_mesh.process(rgb)

        status = "Not Looking Left"
        color = (0, 0, 255)

        if result.multi_face_landmarks:
            face = result.multi_face_landmarks[0]
            gaze_ratio, nose_ratio = check_ratios(face)
            
            is_left = nose_ratio < 0.30

            if is_left:
                status = "Looking Left! 💅"
                color = (0, 255, 0)

                if success_time is None:
                    success_time = time.time()

        cv2.putText(frame, status, (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
        cv2.imshow("Left Checker", frame)

        if success_time is not None:
            if time.time() - success_time >= 1.5:
                print("Held it for 1.5 seconds! We out ✌️")
                cap.release()
                cv2.destroyAllWindows()
                return True

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    return False

def detect_head_right(TIME_LIMIT):
    print("Booting camera for Right Detection... 💀")
    cap = cv2.VideoCapture(0)
    mp_face_mesh = mp.solutions.face_mesh
    face_mesh = mp_face_mesh.FaceMesh(refine_landmarks=True, min_detection_confidence=0.5, min_tracking_confidence=0.5)

    success_time = None
    start_time = time.time()

    while time.time() - start_time < TIME_LIMIT:
        ret, frame = cap.read()
        if not ret: break

        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = face_mesh.process(rgb)

        status = "Not Looking Right"
        color = (0, 0, 255)

        if result.multi_face_landmarks:
            face = result.multi_face_landmarks[0]
            gaze_ratio, nose_ratio = check_ratios(face)
            
            is_right = nose_ratio > 0.65

            if is_right:
                status = "Looking Right! 💅"
                color = (0, 255, 0)

                if success_time is None:
                    success_time = time.time()

        cv2.putText(frame, status, (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
        cv2.imshow("Right Checker", frame)

        if success_time is not None:
            if time.time() - success_time >= 1.5:
                print("Held it for 1.5 seconds! We out ✌️")
                cap.release()
                cv2.destroyAllWindows()
                return True

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    return False

def detect_head_straight(TIME_LIMIT):
    print("Booting camera for Straight Detection... 💀")
    cap = cv2.VideoCapture(0)
    mp_face_mesh = mp.solutions.face_mesh
    face_mesh = mp_face_mesh.FaceMesh(refine_landmarks=True, min_detection_confidence=0.5, min_tracking_confidence=0.5)

    success_time = None
    start_time = time.time()

    while time.time() - start_time < TIME_LIMIT:
        ret, frame = cap.read()
        if not ret: break

        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = face_mesh.process(rgb)

        status = "Not Looking Straight"
        color = (0, 0, 255)

        if result.multi_face_landmarks:
            face = result.multi_face_landmarks[0]
            gaze_ratio, nose_ratio = check_ratios(face)
            
            is_straight = 0.3 <= nose_ratio <= 0.65

            if is_straight:
                status = "Looking Straight! 💅"
                color = (0, 255, 0)

                if success_time is None:
                    success_time = time.time()

        cv2.putText(frame, status, (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
        cv2.imshow("Straight Checker", frame)

        if success_time is not None:
            if time.time() - success_time >= 1.5:
                print("Held it for 1.5 seconds! We out ✌️")
                cap.release()
                cv2.destroyAllWindows()
                return True

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    if success_time is not None:
        cap.release()
        cv2.destroyAllWindows()
        return True

    cap.release()
    cv2.destroyAllWindows()
    return False

In [25]:
def count_blinks(TIME_LIMIT):
    EAR_THRESHOLD = 0.22
    MIN_CLOSED_FRAMES = 2

    blink_counter = 0
    closed_frames = 0

    print("Booting camera for Blink Detection... 💀 (Press 'q' to exit and get count)")
    cap = cv2.VideoCapture(0)

    mp_face_mesh = mp.solutions.face_mesh
    face_mesh = mp_face_mesh.FaceMesh(
        refine_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )

    start_time = time.time()

    while time.time() - start_time < TIME_LIMIT:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = face_mesh.process(rgb)

        if result.multi_face_landmarks:
            face = result.multi_face_landmarks[0]
            h, w, _ = frame.shape

            nose = face.landmark[1]
            cheek_left = face.landmark[234]
            cheek_right = face.landmark[454]

            nose_ratio = get_horizontal_ratio(nose, cheek_left, cheek_right)

            is_looking_straight = 0.35 < nose_ratio < 0.65

            left_eye = get_eye_points(face, LEFT_EYE, w, h)
            right_eye = get_eye_points(face, RIGHT_EYE, w, h)

            leftEAR = eye_aspect_ratio(left_eye)
            rightEAR = eye_aspect_ratio(right_eye)
            ear = (leftEAR + rightEAR) / 2

            if is_looking_straight:
                if ear < EAR_THRESHOLD:
                    closed_frames += 1
                else:
                    if closed_frames >= MIN_CLOSED_FRAMES:
                        blink_counter += 1
                    closed_frames = 0
            else:
                closed_frames = 0

            status_color = (0, 255, 0) if is_looking_straight else (0, 0, 255)
            cv2.putText(frame, f"EAR : {ear:.2f}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, status_color, 2)
        else:
            closed_frames = 0
            cv2.putText(frame, "No Face 💀", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

        cv2.putText(frame, f"Blinks : {blink_counter}", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.imshow("Blink Detection", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

    return blink_counter

In [26]:
def randomChallenge():
    challenges = [
        {
            "text": "Blink 2 times",
            "type": "blink",
            "target": 2
        },
        {
            "text": "Turn your head left",
            "type": "left"
        },
        {
            "text": "Turn your head right",
            "type": "right"
        },
        {
            "text": "Look straight",
            "type": "straight"
        }
    ]

    challenge = random.choice(challenges)
    print(challenge["text"])

    success = False
    TIME_LIMIT = 5

    if challenge["type"] == "blink":
        if count_blinks(TIME_LIMIT) >= challenge["target"]:
            success = True
    elif challenge["type"] == "left":
        if detect_head_left(TIME_LIMIT):
            success = True
    elif challenge["type"] == "right":
        if detect_head_right(TIME_LIMIT):
            success = True
    elif challenge["type"] == "straight":
        if detect_head_straight(TIME_LIMIT):
            success = True
    return success

# Alias to keep it friendly and robust
def random_challenge():
    return randomChallenge()

In [28]:
random_challenge()

Blink 2 times
Booting camera for Blink Detection... 💀 (Press 'q' to exit and get count)


I0000 00:00:1785127022.126555   22808 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1785127022.128196   24101 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.2), renderer: Mesa Intel(R) UHD Graphics (CML GT2)
W0000 00:00:1785127022.130083   24098 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1785127022.141663   24093 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


True